# DroneML – Google Colab Setup

Landmine detection from thermal UAV imagery using deep learning.
This notebook sets up the DroneML project in Colab and lets you run
all phases: preprocessing, training (MobileNetV3/ResNet50/EfficientNetB0),
evaluation, Grad-CAM, and error analysis.

**Before starting:**
1. Go to **Runtime → Change runtime type → T4 GPU**
2. This notebook assumes you have a dataset (or will upload one).

## 1. Mount Google Drive
All data, models, and results persist in your Drive so they survive
across Colab sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the repository
Replace `YOUR_GITHUB_URL` with your repo URL. If you already uploaded
the code to Drive, skip this and copy it from there instead.

In [ ]:
# Option A: Clone from GitHub
!git clone https://github.com/SaranGoutham/DroneML.git /content/DroneML

# Option B: Copy from Google Drive (uncomment and adjust path)
# !cp -r /content/drive/MyDrive/DroneML /content/DroneML

%cd /content/DroneML

## 3. Install dependencies
TensorFlow is pre-installed in Colab, but we need specific versions
and additional packages.

In [ ]:
!pip install -r requirements.txt
!pip install tensorboard pydot graphviz  # optional: diagrams & logs

## 4. Setup project paths (Colab override)

The original config uses `Path(__file__).resolve().parents[2]` which
works when running `python -m src.models.train`. In Colab we override
the project root so notebooks/scripts find the right directories.

We also create a `colab_config.py` that patches the paths and symlink
the results to Drive so they persist.

**Important for dataset:** The project expects thermal mine images in
`dataset/DatainBrief_dataset_1/`. You need to upload your dataset there,
or point to it in Drive.

In [ ]:
import sys, os
from pathlib import Path

# Set project root
PROJECT_ROOT = Path('/content/DroneML').resolve()
os.chdir(str(PROJECT_ROOT))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Symlink results/ to Drive so they persist
DRIVE_RESULTS = Path('/content/drive/MyDrive/DroneML_results')
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

results_dir = PROJECT_ROOT / 'results'
if results_dir.exists():
    !rm -rf "{results_dir}"
!ln -s "{DRIVE_RESULTS}" "{results_dir}"

print(f'Project root: {PROJECT_ROOT}')
print(f'Results symlinked to Drive: {DRIVE_RESULTS}')
print('Setup complete!')

## 5. Dataset preparation

If your dataset is on Google Drive, copy it now:

```
!cp -r /content/drive/MyDrive/DatainBrief_dataset_1 /content/DroneML/dataset/
```

Then run the preprocessing pipeline to scan, convert, enhance, and
split the dataset. This creates `processed_dataset/train/validation/test`.

In [ ]:
# Check if dataset exists
dataset_path = Path('/content/DroneML/dataset/DatainBrief_dataset_1')
if dataset_path.exists():
    print(f'Dataset found: {dataset_path}')
    print(f'Contents: {list(dataset_path.iterdir())[:10]}')
else:
    print('Dataset NOT found.')
    print('Upload your dataset to /content/DroneML/dataset/DatainBrief_dataset_1/')
    print('or symlink it from Drive.')

---
## Preprocessing Pipeline

Run these steps to scan, verify, enhance, and split the raw dataset.
Adjust as needed if parts are already done.

In [ ]:
!python -m src.preprocessing.dataset_loader

In [ ]:
# Convert TIFF -> PNG with CLAHE / gamma enhancement
!python -m src.preprocessing.preprocess_tiff

In [ ]:
# Split by flight date into train / validation / test
!python -m src.preprocessing.split_by_flight

In [ ]:
# (Optional) Offline augmentation to balance classes
!python -m src.preprocessing.augment_offline

In [ ]:
# Build tf.data.Dataset and verify batches
!python -m src.preprocessing.create_tf_dataset

---
## Training

Choose one:
1. **Train a single model** (MobileNetV3 baseline) – quick test
2. **Train all 3 models** (MobileNetV3 + ResNet50 + EfficientNetB0)
   with frozen backbone + fine-tuning phases

In [ ]:
# Option A: Train just MobileNetV3-Large baseline
!python -m src.models.train

In [ ]:
# Option B: Train all 3 models (frozen + fine-tune, ~hours on T4)
!python -m src.models.train_all

---
## Evaluation & Analysis

After training, run these to analyze results.

In [ ]:
# Plot training curves
!python -m src.evaluation.plot_training

In [ ]:
# Confusion matrices, ROC curves, metrics
!python -m src.evaluation.metrics

In [ ]:
# Grad-CAM heatmaps for model explainability
!python -m src.evaluation.gradcam

In [ ]:
# Stratified error analysis (flight, altitude, depth, zone)
!python -m src.evaluation.error_analysis

---
## View Results

All outputs are saved to `results/` (symlinked to Drive) and can be
browsed in the Colab file browser or directly from Drive.

In [ ]:
import subprocess
subprocess.run(['find', '/content/DroneML/results', '-maxdepth', '3', '-type', 'f'],
               capture_output=True, text=True).stdout[:3000]

---
## Troubleshooting

### Path errors (`ModuleNotFoundError: No module named 'src'`)
Run the setup cell in Section 4 again.

### Dataset not found
The dataset must be placed at `dataset/DatainBrief_dataset_1/`.
Upload it to Drive and copy/symlink it into place.

### Out of memory (OOM) with T4 GPU
- Reduce batch size in `src/models/config.py` (change `BATCH_SIZE = 32` to 16)
- Use `train.py` (single model) instead of `train_all.py`

### TensorBoard
```
%load_ext tensorboard
%tensorboard --logdir /content/DroneML/results/logs/tensorboard
```

### All results are in Drive
`/content/drive/MyDrive/DroneML_results/` — they persist after
runtime disconnect.